In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from numpy.random import uniform

In [ ]:
hallway = np.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 0])

In [ ]:
x = np.random.randint(10,size=1000)

In [ ]:
plt.hist(x)

In [ ]:
def search_sorted(cumulative_sum, values):
  # This function is equivalent to np.searchsorted - which is much faster
  indices = []
  for v in values:
    # Assume that the cumulative sum is already sorted properly
    # We return the first index where a[i-1] < v <= a[i]
    for i in range(cumulative_sum.shape[0]):
      if v <= cumulative_sum[i]:
        indices.append(i)
        break
  return np.array(indices)

def simple_resample(particles, weights):
    N = len(particles)
    cumulative_sum = np.cumsum(weights / np.sum(weights))
    cumulative_sum[-1] = 1. # avoid round-off error
    indexes = search_sorted(cumulative_sum, uniform(0,1,size=N))
    # resample according to indexes
    return particles[indexes]


In [ ]:
def sensor_model(z, samples, map):
  # Return the probability of a specific sensor reading given the map
  #      | map=0| map=1|
  #   z=0| 0.75 | 0.10 |
  #   z=1| 0.25 | 0.90 |
  sensor_probs = np.array([[0.75,0.10],[0.25,0.9]])
  predicted_measurements = np.array(map)[samples]
  z_extended = z*np.ones_like(samples)
  results = sensor_probs[z_extended,predicted_measurements]
  return results

In [ ]:
x = 0 # The actual state of the system
z = 1 # The actual measurement of the system
samples = np.random.randint(10,size=1000)
# Compute the likelihood of each sample based upon the measurement
p = sensor_model(z, samples, hallway) # p(y|x,m)
# Use the likelihood to resample
samples2 = simple_resample(samples,p)
plt.hist(samples)
plt.hist(samples2)

In [ ]:
samples = samples2
p = sensor_model(z, samples, hallway) # p(y|x,m)
samples2 = simple_resample(samples,p)
plt.hist(samples)
plt.hist(samples2)

In [ ]:
def predict_move_convolution(particles, action, kernel):
    N = len(particles)
    kN = len(kernel)
    width = int((kN - 1) / 2)
    particles = particles.copy()
    kernel_options = np.arange(len(kernel))-int(kN/2)
    particles += action+np.random.choice(kernel_options,size=N,p=kernel)
    particles %= 10
    return particles

In [ ]:
def particle_filter_1d(x, samples, map, motion_model, sensor_model, resample, action=1):
    # Apply motion to the true robot (for simplicity it is deterministic)
    x = ( x + action ) % len(map)
    # Take a measurement  (for simplicity it is deterministic)
    z = np.array(map)[x]
    # Apply the motion to the samples
    samples = motion_model(samples, action)
    # Compute the likelihood
    p = sensor_model(z, samples, map)
    # Compute the importances and resample
    new_samples = resample(samples, p)
    return new_samples, x

In [ ]:
# @title
def create_basic_bar_plot(counts, point = None, title='Samples Bar Plot', xlabel='Position', ylabel='Count', color='blue'):
    """
    Creates a bar plot of counts using Matplotlib.

    Args:
        counts (numpy.array): A 1D numpy array of counts.
        title (str): The title of the plot.
        xlabel (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
        color (str): The color of the bars.
    """

    # Create an array of indices to represent the categories
    indices = np.arange(len(counts))

    # Create the bar plot
    plt.bar(indices, counts, color=color)

    # Add an indicator for the current state
    if point is not None:
      plt.scatter(point,counts[point], color='orange')
    # Set the title and axis labels
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    # Set the x-axis ticks and labels
    plt.xticks(indices)

    # Show the plot
    plt.show()

In [ ]:
from ipywidgets import interact, IntSlider

def make_simulator_plot(x, map, samples, motion_model, sensor_model, resample, timesteps=20):
  sample_counts = [np.histogram(samples)[0]]
  states = [x]
  for _ in range(timesteps):
    samples, x = particle_filter_1d(x, samples, map, motion_model, sensor_model, resample)
    sample_counts.append(np.histogram(samples)[0])
    states.append(x)
  def simulator(time_step):
    create_basic_bar_plot(sample_counts[time_step], states[time_step])
    plt.show()
  return simulator

In [ ]:
samples = np.random.randint(10,size=1000)
x_init = 1
timesteps = 20
hallway = np.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 0])
simulate = make_simulator_plot(
    x=x_init,
    map=hallway,
    samples=samples,
    motion_model=lambda x,u: predict_move_convolution(x,u,kernel=[0.1,0.8,0.1]),
    sensor_model=sensor_model,
    resample=simple_resample,
    timesteps=timesteps)

interact(simulate, time_step=IntSlider(value=0, max=timesteps));